# HealthyLife Policy Assistant — RAG Demo

**A. Indexing:** PDF → Chunking → Embedding → MongoDB Atlas Vector Search

Setup: `pip install -r requirements.txt`, copy `.env.example` to `.env` and fill in `MONGODB_URI`.

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv(override=True)  # re-read .env even if a var was already set (e.g. empty)
MONGODB_URI = os.environ["MONGODB_URI"]
PDF_PATH = "data/healthylife_policy_handbook.pdf"
DB_NAME, COLLECTION, INDEX_NAME = "healthylife_rag", "policy_chunks", "vector_index"
EMBED_MODEL = "sentence-transformers/all-MiniLM-L6-v2"

## A. Indexing pipeline
### A1. Load the PDF (one Document per page)

In [ ]:
%pip install -q pypdf langchain-core
from pypdf import PdfReader
from langchain_core.documents import Document

pages = [
    Document(page_content=p.extract_text(), metadata={"source": PDF_PATH, "page": i + 1})
    for i, p in enumerate(PdfReader(PDF_PATH).pages)
]
print(f"{len(pages)} pages, {sum(len(p.page_content) for p in pages)} characters")

### A2. Chunking
Recursive splitting tries paragraph → line → sentence → word boundaries, so chunks stay readable. The overlap keeps sentences that sit on a boundary retrievable.

In [ ]:
%pip install -q langchain-text-splitters
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=120)
chunks = splitter.split_documents(pages)
print(f"{len(chunks)} chunks")
print("--- sample chunk ---")
print(chunks[4].page_content)

### A3. Embedding
`all-MiniLM-L6-v2` runs locally (no API key) and turns text into a 384-dimensional vector.

In [ ]:
%pip install -q langchain-huggingface sentence-transformers

from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(model_name=EMBED_MODEL, encode_kwargs={"normalize_embeddings": True})
DIMS = len(embeddings.embed_query("hello"))
print("embedding dimensions:", DIMS)

### A4. Store in MongoDB and create the vector index
Each chunk becomes one document: `text`, `embedding`, `page`, `source`. Re-running this cell replaces the old chunks.

In [ ]:
%pip install -q pymongo langchain-mongodb
from pymongo import MongoClient
from langchain_mongodb import MongoDBAtlasVectorSearch

client = MongoClient(MONGODB_URI)
collection = client[DB_NAME][COLLECTION]
vector_store = MongoDBAtlasVectorSearch(
    collection=collection, embedding=embeddings, index_name=INDEX_NAME, relevance_score_fn="cosine"
)

collection.delete_many({})
vector_store.add_documents(chunks)
print(collection.count_documents({}), "chunks stored in", f"{DB_NAME}.{COLLECTION}")

if not any(ix["name"] == INDEX_NAME for ix in collection.list_search_indexes()):
    vector_store.create_vector_search_index(dimensions=DIMS, wait_until_complete=120)
    print("created vector index", INDEX_NAME)
else:
    print("vector index", INDEX_NAME, "already exists")

# Atlas Search indexes new writes asynchronously: wait until every chunk is searchable
import time
for _ in range(60):
    if len(vector_store.similarity_search("policy", k=len(chunks))) == len(chunks):
        break
    time.sleep(1)
else:
    raise TimeoutError("vector index did not catch up within 60 s")
print("all chunks searchable")